<a href="https://colab.research.google.com/github/xyt556/I-GUIDE-GeoAI-Education/blob/main/notebooks/05-object-detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 目标检测

## 引言

图像识别为整个图像分配一个单一标签，语义分割标记每个像素，但两者都无法区分单个对象。目标检测通过生成带有类别标签和置信度分数的边界框来弥补这一差距，以在图像中定位和识别单个对象。

本教程涵盖了地理空间图像目标检测的基础知识。您将探索主要的架构家族，使用 `geoai` 包在 [NWPU-VHR-10](https://data.source.coop/opengeos/geoai/NWPU-VHR-10.zip) 基准数据集上训练 Faster R-CNN v2 检测器，使用 COCO 风格的指标评估结果，在新图像上运行推理，并将训练好的模型发布到 Hugging Face Hub。

## 学习目标

通过本教程，您将能够：

- 解释目标检测与图像分类和语义分割有何不同
- 描述边界框、置信度分数、非极大值抑制和锚框
- 比较两阶段、单阶段、基于 Transformer 和零样本检测架构
- 准备 COCO 注释格式的检测数据集
- 使用 `geoai` 包训练多类别目标检测模型
- 使用 COCO 风格的平均精度均值 (mAP) 评估检测性能
- 在新图像上运行推理并可视化检测结果
- 将训练好的模型发布到 Hugging Face Hub 并从托管模型运行推理

## 理解目标检测

### 分类与检测

图像分类为整个图像分配一个单一标签，但没有说明特定对象在哪里或有多少个。目标检测生成可变数量的边界框，每个边界框都带有类别标签和置信度分数，从而实现单个特征的计数、定位和映射。

语义分割提供像素级标签，但不分离单个对象。目标检测提供对象级定位，但不描绘确切的边界。实例分割结合了这两种能力。

### 关键概念

**边界框**是检测模型的基本输出，由指定对象周围矩形的四个坐标定义。在地理空间应用中，这些像素空间坐标通过图像的仿射变换转换为地理坐标。

**置信度分数**伴随每个边界框，范围从 0 到 1，表示模型对预测的确定性。高阈值减少假阳性但可能漏掉对象，而低阈值捕获更多对象但引入更多假检测。

**交并比 (IoU)** 衡量预测边界框与真实框的重叠程度，计算为重叠面积除以并集面积。它在训练和评估期间都会使用。

**非极大值抑制 (NMS)** 是一种后处理步骤，通过为每个对象保留置信度最高的框来删除冗余的重叠检测。这对于在密集场景中生成清晰输出至关重要。

**锚框**是具有各种比例和纵横比的预定义边界框模板，作为预测的起点。模型学习相对于这些锚框的小偏移量，而不是从头开始预测坐标。

## 检测架构

### 两阶段检测器

两阶段检测器将检测分为候选区域生成和分类。**Faster R-CNN** 首先使用区域提议网络 (RPN) 来提议候选区域，然后通过专用头部对每个提议进行分类和细化。

两阶段检测器实现了高精度并能很好地处理多尺度，使其适用于地理空间图像。然而，它们的顺序性质使其比单阶段替代品慢。

### 单阶段检测器

单阶段检测器在一次通过中直接从特征图预测边界框和类别标签，使其速度显著加快。

- **YOLO** (You Only Look Once) 将检测视为一个密集预测问题，并已发展成为一个平衡速度和精度的模型家族。
- **SSD** (Single Shot MultiBox Detector) 从不同分辨率的多个特征图进行预测，检测不同尺度的对象。
- **RetinaNet** 使用焦点损失函数通过将训练集中在难样本上来解决类别不平衡问题。`geoai` 通过 `retinanet_resnet50_fpn_v2` 支持。
- **FCOS** 是一种无锚检测器，直接在每个空间位置预测边界框。`geoai` 通过 `fcos_resnet50_fpn` 支持。

### 基于 Transformer 的检测器

[DETR](https://huggingface.co/docs/transformers/model_doc/detr) (DEtection TRansformer) 使用 Transformer 编码器-解码器架构将目标检测视为一个集合预测问题，消除了对锚框和 NMS 的需求。其主要优点是简洁性和全局上下文推理，尽管它收敛较慢，并且可能难以处理非常小的对象。

### 零样本检测

零样本检测模型无需任务特定训练数据即可识别文本提示描述的对象。[OWL-ViT](https://huggingface.co/docs/transformers/en/model_doc/owlvit) 将视觉 Transformer 与文本编码器相结合，通过自然语言描述（如“太阳能电池板”或“游泳池”）实现检测。

[Grounding DINO](https://huggingface.co/docs/transformers/en/model_doc/grounding-dino) 扩展了这一范式，将基于 DINO 的检测与基础语言理解相结合，实现了强大的零样本性能。

### 选择架构

架构的选择取决于您的应用程序需求：

- 当精度是首要任务且推理速度不那么关键时，**Faster R-CNN** 是一个很好的默认选择。它能很好地处理多尺度对象，是 `geoai` 包检测管道中的默认架构。
- 当处理速度很重要时，例如扫描大量卫星图像档案或支持近实时监控应用程序时，首选 **YOLO**。
- **DETR 及其变体**非常适合需要全局上下文推理的场景，以及当您希望避免调整锚框配置时。
- **零样本检测器** (OWL-ViT, Grounding DINO) 是探索性分析、快速原型设计或缺乏标注训练数据的应用的理想选择。

`geoai` 包通过 `model_name` 参数支持多种检测架构：

| 模型名称                            | 类型         | 备注                                     |
| ----------------------------------- | ------------ | ---------------------------------------- |
| `fasterrcnn_resnet50_fpn_v2`        | 两阶段       | 默认，良好的精度/速度权衡                |
| `fasterrcnn_mobilenet_v3_large_fpn` | 两阶段       | 最快的两阶段选项                         |
| `retinanet_resnet50_fpn_v2`         | 单阶段       | 快速，能很好地处理类别不平衡             |
| `fcos_resnet50_fpn`                 | 单阶段       | 无锚                                     |
| `maskrcnn_resnet50_fpn`             | 两阶段       | 还生成实例掩码（最慢）                   |

在实践中，许多项目从预训练或零样本模型开始评估可行性，然后转向特定任务检测器进行生产。将检测器的优势与目标对象的规模匹配通常比选择最新的架构更重要。

## 准备检测数据集

### 标注格式

**COCO 格式**将标注存储在一个 JSON 文件中，其中包含图像元数据、类别定义和边界框（x、y、宽度、高度），以绝对像素坐标表示。它是 `geoai` 包使用的格式。

**YOLO 格式**为每张图像使用一个文本文件，每行描述一个对象，格式为 `class_id center_x center_y width height`，以归一化坐标表示。

### NWPU-VHR-10 数据集

[NWPU-VHR-10](https://data.source.coop/opengeos/geoai/NWPU-VHR-10.zip) 数据集是高分辨率遥感图像目标检测中广泛使用的基准。它包含 800 张图像（650 张正样本，150 张仅背景）涵盖 10 个类别：飞机、船舶、储油罐、棒球场、网球场、篮球场、操场、港口、桥梁和车辆。

`geoai` 包提供 `prepare_nwpu_vhr10` 函数，自动将带标注的图像分成训练集和验证集，并生成 COCO 格式的标注文件。

## 评估检测结果

### 平均精度均值 (mAP)

如果检测与真实框的 IoU 超过某个阈值且类别正确，则为**真阳性**；否则为**假阳性**。未匹配的真实框为**假阴性**。

单个类别的**平均精度 (AP)** 是精确率-召回率曲线下的面积。**平均精度均值 (mAP)** 是所有类别的 AP 平均值。

### 精确率-召回率曲线

精确率-召回率曲线随着置信度阈值的变化绘制精确率与召回率。一个强大的检测器即使在召回率很高时也能保持高精确率。检查每个类别的曲线可以揭示模型在哪些对象类型上表现不佳。

### IoU 阈值

- **mAP@0.5** 需要至少 50% 的重叠，在地理空间应用中很常见，其中近似定位就足够了。
- **mAP@0.75** 需要 75% 的重叠，奖励更精确的定位。
- **mAP@0.5:0.95** 在 0.5 到 0.95 的阈值范围内以 0.05 的步长取平均值，是主要的 COCO 基准指标。

## 安装

取消注释以下行以安装所需的包。

## 导入库

`geoai` 包提供了用于完整目标检测管道的功能，包括数据集准备、训练、评估、推理和模型共享。

## 下载 NWPU-VHR-10 数据集

NWPU-VHR-10 数据集以 zip 存档的形式托管在 Source Cooperative 上。`download_file` 工具会自动下载并解压。

## 探索数据集

数据集包含 10 个对象类别以及索引为 0 的背景类别。

## 准备数据集

`prepare_nwpu_vhr10` 函数将正样本图像分割成训练集和验证集，并为每个分割生成 COCO 格式的标注文件。

## 可视化样本标注

在训练之前，可视化带有其真实边界框的样本图像，以验证标注。

## 训练多类别检测模型

`train_multiclass_detector` 函数处理完整的训练管道：加载数据、构建模型、运行训练循环和保存最佳检查点。关键参数包括模型架构、类别名称、训练轮次、批次大小和学习率。

## 绘制训练指标

绘制训练损失、验证 IoU 和学习率调度随训练轮次的变化，以评估模型收敛情况。

## 使用 COCO 指标进行评估

`evaluate_multiclass_detector` 函数计算验证集上的 COCO 风格 mAP 指标，包括每个类别的 AP 分数。

大型、独特的物体（如篮球场和网球场）比小型或更模糊的物体（如车辆和储油罐）获得更高的 AP。

## 在样本图像上运行推理

`multiclass_detection` 函数处理切片、模型推理、跨切片边界的 NMS 和结果组装。

## 可视化检测结果

`visualize_multiclass_detections` 函数将检测到的边界框叠加在原始图像上，按类别着色并用置信度分数标记。

## 对多张图像进行批量推理

`batch_multiclass_detection` 函数对多张图像运行推理并生成可视化网格。

预测并不完美。您可能会在结果中发现假阳性和假阴性。

## 发布和重用模型

### 推送到 Hugging Face Hub

通过 Hugging Face Hub 共享训练好的模型，协作人员无需原始训练数据或计算资源即可运行推理。

### 从 Hub 运行推理

`predict_detector_from_hub` 函数从 Hub 下载模型并自动运行推理，无需本地检查点或类别名称列表。

## 关键要点

1. 目标检测利用边界框对单个对象进行定位和分类，填补了图像分类和像素级分割之间的空白。

2. 边界框、IoU、NMS 和锚框是检测模型生成、优化和过滤预测的基础概念。

3. 两阶段检测器（Faster R-CNN）优先考虑精度，单阶段检测器（YOLO、RetinaNet、FCOS）优先考虑速度，基于 Transformer 的检测器（DETR）简化了管道，而零样本检测器（OWL-ViT、Grounding DINO）消除了对任务特定训练数据的需求。

4. NWPU-VHR-10 数据集为遥感中的多类别目标检测提供了一个标准的 10 类别基准。

5. `geoai` 包简化了从数据集准备到训练、评估、推理和模型共享的完整检测工作流程。

6. COCO 风格的 mAP 指标在多个 IoU 阈值下提供了全面的评估，每个类别的 AP 揭示了模型的优缺点。

7. 置信度阈值调整平衡了精确率和召回率，最佳阈值取决于假阳性和假阴性的应用特定成本。

8. 将模型发布到 Hugging Face Hub 可以实现共享和重用，而无需本地训练基础设施。

In [ ]:
# %pip install -U "geoai-py[extra]"

## Import Libraries

The `geoai` package provides functions for the full object detection pipeline, including dataset preparation, training, evaluation, inference, and model sharing.

In [ ]:
import os
import json

import geoai

## Download the NWPU-VHR-10 Dataset

The NWPU-VHR-10 dataset is available as a zip archive hosted on Source Cooperative. The `download_file` utility downloads and extracts it automatically.

In [ ]:
url = "https://data.source.coop/opengeos/geoai/NWPU-VHR-10.zip"
data_dir = geoai.download_file(url)

In [ ]:
print(f"Dataset directory: {data_dir}")
print(f"Contents: {os.listdir(data_dir)}")

## Explore the Dataset

The dataset contains 10 object classes plus a background class at index 0.

In [ ]:
print(f"\nNWPU-VHR-10 Classes:")
for i, name in enumerate(geoai.NWPU_VHR10_CLASSES):
    print(f"  {i}: {name}")

```text
NWPU-VHR-10 类别：
  0: 背景
  1: 飞机
  2: 船舶
  3: 储油罐
  4: 棒球场
  5: 网球场
  6: 篮球场
  7: 操场
  8: 港口
  9: 桥梁
  10: 车辆
```

## Prepare the Dataset

The `prepare_nwpu_vhr10` function splits the positive images into training and validation sets and generates COCO-format annotation files for each split.

In [ ]:
splits = geoai.prepare_nwpu_vhr10(data_dir, val_split=0.2, seed=42)

In [ ]:
print(f"Images directory: {splits['images_dir']}")
print(f"Number of classes: {splits['num_classes']}")
print(f"Class names: {splits['class_names']}")
print(f"Training images: {len(splits['train_image_ids'])}")
print(f"Validation images: {len(splits['val_image_ids'])}")

```text
图像目录: ./NWPU-VHR-10/positive_image_set
类别数量: 11
类别名称: ['背景', '飞机', '船舶', '储油罐', '棒球场', '网球场', '篮球场', '操场', '港口', '桥梁', '车辆']
训练图像: 509
验证图像: 128
```

## Visualize Sample Annotations

Visualize sample images with their ground truth bounding boxes to verify annotations before training.

In [ ]:
geoai.visualize_coco_annotations(
    annotations_path=splits["annotations_path"],
    images_dir=splits["images_dir"],
    num_samples=6,
    random=True,
    seed=1,
    cols=3,
    figsize=(12, 6),
)

## Train a Multi-Class Detection Model

The `train_multiclass_detector` function handles the full training pipeline: loading data, constructing the model, running the training loop, and saving the best checkpoint. Key parameters include the model architecture, class names, number of epochs, batch size, and learning rate.

In [ ]:
output_dir = "nwpu_output"

model_path = geoai.train_multiclass_detector(
    images_dir=splits["images_dir"],
    annotations_path=splits["train_annotations"],
    output_dir=output_dir,
    model_name="fasterrcnn_resnet50_fpn_v2",
    class_names=splits["class_names"],
    num_channels=3,
    batch_size=4,
    num_epochs=10,
    learning_rate=0.005,
    val_split=0.1,
    seed=42,
    pretrained=True,
    verbose=True,
)

## Plot Training Metrics

Plot training loss, validation IoU, and learning rate schedule over epochs to assess model convergence.

In [ ]:
geoai.plot_detection_training_history(
    history_path=os.path.join(output_dir, "training_history.pth"),
)

## Evaluate with COCO Metrics

The `evaluate_multiclass_detector` function computes COCO-style mAP metrics on the validation set, including per-class AP scores.

In [ ]:
metrics = geoai.evaluate_multiclass_detector(
    model_path=model_path,
    images_dir=splits["images_dir"],
    annotations_path=splits["val_annotations"],
    num_classes=splits["num_classes"],
    class_names=splits["class_names"][1:],  # Exclude background
    batch_size=4,
)

```text
评估结果：
  mAP@0.5:        0.7312
  mAP@0.75:       0.4936
  mAP@[0.5:0.95]: 0.4428

  AP@0.5/每类别：
    AP@0.5/飞机: 0.7106
    AP@0.5/棒球场: 0.7885
    AP@0.5/篮球场: 0.8957
    AP@0.5/桥梁: 0.9052
    AP@0.5/操场: 0.7081
    AP@0.5/港口: 0.5322
    AP@0.5/船舶: 0.6349
    AP@0.5/储油罐: 0.5624
    AP@0.5/网球场: 0.8967
    AP@0.5/车辆: 0.6781
```

大型、独特的物体（如篮球场和网球场）比小型或更模糊的物体（如车辆和储油罐）获得更高的 AP。

In [ ]:
# Load validation data to pick a test image
with open(splits["val_annotations"], "r") as f:
    val_data = json.load(f)

test_img_info = val_data["images"][0]
test_img_path = os.path.join(splits["images_dir"], test_img_info["file_name"])
print(f"Test image: {test_img_path}")

In [ ]:
output_raster = "nwpu_detection_output.tif"

result_path, inference_time, detections = geoai.multiclass_detection(
    input_path=test_img_path,
    output_path=output_raster,
    model_path=model_path,
    num_classes=splits["num_classes"],
    class_names=splits["class_names"],
    window_size=512,
    overlap=256,
    confidence_threshold=0.5,
    batch_size=4,
    num_channels=3,
)

print(f"\nInference time: {inference_time:.2f}s")
print(f"Total detections: {len(detections)}")

```text
NMS 前收集到 27 次检测
NMS 后：7 次检测
多类别检测在 0.14 秒内完成
最终检测：7 次
  港口：6 次检测
  桥梁：1 次检测
多类别检测结果已保存到 nwpu_detection_output.tif

推理时间：0.14 秒
总检测次数：7 次
```

## Visualize Detections

The `visualize_multiclass_detections` function overlays detected bounding boxes on the original image, colored by class and labeled with confidence scores.

In [ ]:
geoai.visualize_multiclass_detections(
    image_path=test_img_path,
    detections=detections,
    class_names=splits["class_names"],
    confidence_threshold=0.5,
    figsize=(12, 10),
)

## Batch Inference on Multiple Images

The `batch_multiclass_detection` function runs inference on multiple images and produces a visualization grid.

In [ ]:
val_image_paths = [
    os.path.join(splits["images_dir"], img["file_name"])
    for img in val_data["images"][:4]
]

results = geoai.batch_multiclass_detection(
    image_paths=val_image_paths,
    output_dir="nwpu_batch_output",
    model_path=model_path,
    num_classes=splits["num_classes"],
    class_names=splits["class_names"],
    confidence_threshold=0.5,
    num_channels=3,
    figsize=(16, 12),
)

预测并不完美。您可能会在结果中发现假阳性和假阴性。

## Publish and Reuse Models

### Push to Hugging Face Hub

Sharing a trained model through Hugging Face Hub lets collaborators run inference without the original training data or compute resources.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
url = geoai.push_detector_to_hub(
    model_path=model_path,
    repo_id="your-username/nwpu-vhr10-fasterrcnn",
    model_name="fasterrcnn_resnet50_fpn_v2",
    num_classes=splits["num_classes"],
    class_names=splits["class_names"],
)

### Run Inference from Hub

The `predict_detector_from_hub` function downloads a model from the Hub and runs inference automatically, requiring no local checkpoint or class name list.

In [ ]:
sample_img_path = os.path.join(splits["images_dir"], "608.jpg")

result_path, inference_time, detections = geoai.predict_detector_from_hub(
    input_path=sample_img_path,
    output_path="hub_detection.tif",
    repo_id="giswqs/nwpu-vhr10-fasterrcnn",
    confidence_threshold=0.5,
)

print(f"Inference time: {inference_time:.2f}s")
print(f"Total detections: {len(detections)}")

# Clean up
if os.path.exists("hub_detection.tif"):
    os.remove("hub_detection.tif")

```text
NMS 前收集到 34 次检测
NMS 后：8 次检测
多类别检测在 0.13 秒内完成
最终检测：8 次
  棒球场：1 次检测
  网球场：4 次检测
  篮球场：3 次检测
多类别检测结果已保存到 hub_detection.tif
推理时间：0.13 秒
总检测次数：8 次
```

In [ ]:
geoai.visualize_multiclass_detections(
    image_path=sample_img_path,
    detections=detections,
    class_names=geoai.NWPU_VHR10_CLASSES,
    confidence_threshold=0.5,
    figsize=(12, 10),
)

## Key Takeaways

1. Object detection localizes and classifies individual objects using bounding boxes, filling the gap between image classification and pixel-level segmentation.

2. Bounding boxes, IoU, NMS, and anchor boxes are the foundational concepts underlying how detection models generate, refine, and filter predictions.

3. Two-stage detectors (Faster R-CNN) prioritize accuracy, single-stage detectors (YOLO, RetinaNet, FCOS) prioritize speed, transformer-based detectors (DETR) simplify the pipeline, and zero-shot detectors (OWL-ViT, Grounding DINO) eliminate the need for task-specific training data.

4. The NWPU-VHR-10 dataset provides a standard 10-class benchmark for multi-class object detection in remote sensing.

5. The `geoai` package streamlines the full detection workflow from dataset preparation through training, evaluation, inference, and model sharing.

6. COCO-style mAP metrics at multiple IoU thresholds provide comprehensive evaluation, with per-class AP revealing strengths and weaknesses.

7. Confidence threshold tuning balances precision and recall, with the optimal threshold depending on application-specific costs of false positives versus false negatives.

8. Publishing models to Hugging Face Hub enables sharing and reuse without requiring local training infrastructure.